# Train Task 1: Unisensory Block-Switching CTRNN

Train a CTRNN on Task 1 where the network decides left/right based on either visual or auditory cues. Context (which modality is relevant) switches every 50 trials and is **explicitly cued**.

**Expected behavior:** Accuracy should reach >95% within ~50-80 epochs. No performance dip at context switches because the context cue tells the network which modality to use.

**Key training details:**
- Hidden state persists across trials within a session (not reset between trials)
- Feedback is dynamic: based on the model's actual predictions, not assumed correct
- Loss computed only on response epoch (last 5 timesteps)
- L1 regularization on firing rates promotes sparse representations

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.models import CTRNN
from src.tasks import Task1Session
from src.training import train_model, evaluate, run_session
from src.utils import set_seed, save_checkpoint, load_checkpoint

sns.set_style('whitegrid')
sns.set_context('notebook')

## Configuration

Adjust these hyperparameters as needed. The defaults work well for initial training.

In [ ]:
# --- Hyperparameters ---
SEED = 42
HIDDEN_SIZE = 256
N_EPOCHS = 100
LR = 1e-3
L1_LAMBDA = 1e-4
GRAD_CLIP = 1.0
N_SESSIONS_TRAIN = 8   # sessions per epoch
N_SESSIONS_EVAL = 4    # sessions per evaluation
EVAL_EVERY = 5         # evaluate every N epochs

# Task parameters
N_TRIALS = 300
BLOCK_SIZE = 50

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')

set_seed(SEED)

## Initialize Model

In [ ]:
model = CTRNN(
    input_size=8,
    hidden_size=HIDDEN_SIZE,
    output_size=2,
    dt=20.0,
    tau=100.0,
    sigma_rec=0.05,
    seed=SEED
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'CTRNN: {HIDDEN_SIZE} hidden units, {n_params:,} parameters')
print(f'gamma = {model.gamma}, noise_scale = {model.noise_scale:.4f}')

## Train

Each epoch processes 8 sessions (each with 300 trials). The model receives dynamic feedback based on its own predictions. Training takes ~5-15 minutes depending on hardware.

Watch for:
- Accuracy climbing above 0.9 by epoch ~30-50
- Loss decreasing steadily

In [ ]:
task_kwargs = {
    'n_trials': N_TRIALS,
    'block_size': BLOCK_SIZE,
}

history = train_model(
    model=model,
    task_class=Task1Session,
    task_kwargs=task_kwargs,
    device=DEVICE,
    n_epochs=N_EPOCHS,
    lr=LR,
    grad_clip=GRAD_CLIP,
    n_sessions_train=N_SESSIONS_TRAIN,
    n_sessions_eval=N_SESSIONS_EVAL,
    eval_every=EVAL_EVERY,
    l1_lambda=L1_LAMBDA,
    checkpoint_dir='../checkpoints',
    checkpoint_prefix='task1',
    seed=SEED,
    verbose=True
)

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history['train_loss'], alpha=0.7, label='Train')
axes[0].plot(history['epoch'], history['eval_loss'], 'o-', label='Eval', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history['train_accuracy'], alpha=0.7, label='Train')
axes[1].plot(history['epoch'], history['eval_accuracy'], 'o-', label='Eval', markersize=3)
axes[1].axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_ylim(0.3, 1.05)
axes[1].legend()

plt.suptitle('Task 1 Training', y=1.02)
plt.tight_layout()
plt.savefig('../figures/task1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation: Per-Block Accuracy

Since Task 1 has explicit context cues, accuracy should be uniformly high across all block positions. No performance dip at switch points.

In [ ]:
# Evaluate on fresh sessions
eval_result = evaluate(
    model, Task1Session, task_kwargs, DEVICE,
    n_sessions=8, seed=7777
)
print(f'Overall accuracy: {eval_result["accuracy"]:.3f}')

# Per-trial accuracy by position in session
n_eval_sessions = 8
n_trials = N_TRIALS
trial_acc = np.zeros(n_trials)
trial_count = np.zeros(n_trials)

for i, (correct, meta) in enumerate(zip(eval_result['per_trial_accuracy'],
                                        eval_result['per_trial_metadata'])):
    trial_pos = i % n_trials
    trial_acc[trial_pos] += int(correct)
    trial_count[trial_pos] += 1

trial_acc = trial_acc / np.maximum(trial_count, 1)

# Smooth with sliding window
window = 20
smoothed = np.convolve(trial_acc, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(window-1, n_trials), smoothed, linewidth=1.5)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

# Mark switch points
for sp in range(BLOCK_SIZE, n_trials, BLOCK_SIZE):
    ax.axvline(sp, color='black', linestyle='--', alpha=0.3)

ax.set_xlabel('Trial in Session')
ax.set_ylabel(f'Accuracy (window={window})')
ax.set_title('Task 1: Per-Trial Accuracy (should be flat across switches)')
ax.set_ylim(0.3, 1.05)

plt.tight_layout()
plt.savefig('../figures/task1_per_trial_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Final Checkpoint

This checkpoint will be loaded by Task 2 training for sequential learning.

In [ ]:
# The train_model function already saves checkpoints, but let's also save
# a clearly named one for Task 2 to load
print('Task 1 checkpoint saved at: ../checkpoints/task1_final.pt')
print(f'Final eval accuracy: {eval_result["accuracy"]:.3f}')